In [ ]:
# System packages
!apt-get update -y
!apt-get install -y zstd curl

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

print("Ollama server started")

# Install Python libraries
!pip install ollama chromadb -q

# Pull required models
!ollama pull llama3
!ollama pull mxbai-embed-large

print("Models Pulled Successfully")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:

# FINANCIAL RAG ASSISTANT

import ollama
import chromadb
import time

time.sleep(3)

#  Sample Data:
data = [
    "UPI payment to Amazon for ₹1200. Category: Shopping.",
    "Salary credited from Infosys for ₹75000. Category: Income.",
    "SIP deducted for ₹5000 in SBI Mutual Fund. Category: Investment.",
    "Electricity bill payment for ₹2500. Category: Utilities.",
    "Restaurant dinner for ₹1500. Category: Food & Dining.",
    "Loan EMI deducted for ₹10000. Category: Debt.",
    "Received interest from Fixed Deposit for ₹1500. Category: Income.",
    "Grocery shopping at Reliance Fresh for ₹3000. Category: Groceries."
]

#  Vactore DB Setup:
client = chromadb.PersistentClient(path="./finance_db")
collection = client.get_or_create_collection(name="finance_data")

# Optional: Clear previous entries (prevents duplicate ID error)
try:
    collection.delete(ids=[f"id_{i}" for i in range(len(data))])
except:
    pass

#  Generate Embeddings and store in V_DB:
for i, text in enumerate(data):
    embedding = ollama.embeddings(
        model="mxbai-embed-large",
        prompt=text
    )["embedding"]

    collection.add(
        ids=[f"id_{i}"],
        embeddings=[embedding],
        documents=[text]
    )

print("Data Indexed Successfully.")

#  Query:
query = "How much did I invest?"

query_embedding = ollama.embeddings(
    model="mxbai-embed-large",
    prompt=query
)["embedding"]

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

context = "\n".join(results["documents"][0])

print("\nRetrieved Context:\n", context)

# Response:
response = ollama.generate(
    model="llama3",
    prompt=f"""
You are a financial assistant.

Context:
{context}

Question:
{query}

Answer clearly:
"""
)

print("\n--- FINAL ANSWER ---\n")
print(response["response"])

Data Indexed Successfully.

Retrieved Context:
 SIP deducted for ₹5000 in SBI Mutual Fund. Category: Investment.
Salary credited from Infosys for ₹75000. Category: Income.

--- FINAL ANSWER ---

Based on the given information, you invested ₹5000 through a Systematic Investment Plan (SIP) in SBI Mutual Fund.


In [ ]:
# Sample Queries:

# sample_queries = [
#     "What did I spend on shopping?",
#     "How much salary did I receive?",
#     "What are my utility expenses?",
#     "How much did I spend on food?",
#     "What is my loan EMI payment?",
#     "How much interest did I receive?",
#     "What was my grocery expense?",
#     "List all my income sources.",
#     "Tell me about my recent expenses.",
#     "What was the cost of my restaurant dinner?"
# ]

# for i, q in enumerate(sample_queries):
#     print(f"Sample Query {i+1}: {q}")

